In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_fii_geral_cvm"
NOME_TABELA  = f"silver_cvm_fii_geral" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## Função Para Ler a Partição

## CVM - Fundos Imobiliarios - Geral

In [0]:
df_silver_fii_geral = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_geral = df_silver_fii_geral.withColumn(
    "CNPJ_FUNDO_CLASSE",
    PipelineConfig.normalizar_cnpj("CNPJ_FUNDO_CLASSE")
)

#### 1.1.2 Retirando dados duplicados

Com a entrada em vigor da **Resolução CVM 175**, os Fundos de Investimento Imobiliário (FIIs) passaram a exigir uma separação estrutural entre o **Fundo** (a "casca" jurídica) e a **Classe** (onde o patrimônio, os cotistas e os ativos realmente ficam registrados). 

Durante esta fase de adaptação, identificamos que as administradoras frequentemente enviam o mesmo Informe Mensal em duplicidade para a CVM: um registro classificado como "Fundo" e outro como "Classe", por vezes até com Datas de Entrega diferentes.

**A Regra de Desempate Determinística (Camada Silver):**
Para garantir a unicidade da nossa chave de negócio (`CNPJ_FUNDO_CLASSE` + `Data_Referencia`) sem perder o histórico correto, aplicamos o seguinte ranqueamento:
1. **Prioridade Máxima:** Registros onde `Tipo_Fundo_Classe = 'Classe'`.
2. **Critério de Desempate Extra:** Se houver empate no tipo, priorizamos a `Data_Entrega` mais recente (ordem decrescente).

*Abaixo, um exemplo real tratado pelo pipeline. A primeira linha será preservada (Classe) e a segunda será descartada (Fundo):*

```
+-----------------+------------------+---------------+------+------------+--------------------+
|Tipo_Fundo_Classe| CNPJ_FUNDO_CLASSE|Data_Referencia|Versao|Data_Entrega|   Nome_Fundo_Classe|
+-----------------+------------------+---------------+------+------------+--------------------+
|           Classe|    14056001000162|     2025-10-01|     1|  2025-11-17|PERSONALE I FUNDO...|
|            Fundo|    14056001000162|     2025-10-01|     1|  2025-12-15|PERSONALE I FUNDO...|
+-----------------+------------------+---------------+------+------------+--------------------+
```

In [0]:
# Definimos a chave que deveria ser única
chave_negocio = ["CNPJ_FUNDO_CLASSE", "Data_Referencia"]

# Cria uma "janela de observação" para o grupo (CNPJ + Data)
window_tipo = Window.partitionBy(chave_negocio)

# Mapeia se, dentro daquele dia e CNPJ, a CVM enviou pelo menos um registro como "Classe"
df_silver_fii_geral = df_silver_fii_geral.withColumn(
    "_grupo_tem_classe", 
    f.max(f.when(f.col("Tipo_Fundo_Classe") == "Classe", f.lit(1)).otherwise(f.lit(0))).over(window_tipo)
)

# Filtro Silencioso: 
# Mantém a linha SE ela for 'Classe' OU SE for 'Fundo' num grupo que ainda não tem 'Classe'
df_silver_fii_geral = df_silver_fii_geral.filter(
    (f.col("Tipo_Fundo_Classe") == "Classe") | 
    ((f.col("Tipo_Fundo_Classe") == "Fundo") & (f.col("_grupo_tem_classe") == 0))
).drop("_grupo_tem_classe")

# Chama a função que já criamos no config.py
df_silver_fii_geral, df_quarentena_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_silver_fii_geral,
    chave_negocio=chave_negocio,
    coluna_ordenacao="Data_Entrega" # O desempate para a Silver (pega a data de entrega mais recente)
)

# Salva as linhas INJUSTIFICÁVEIS na tabela de quarentena (que depois vai gerar o e-mail)
PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_duplicadas, 
    tabela_origem="bronze_fii_geral_cvm", 
    data_proc=DATA_PROC
)


#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "CNPJ_FUNDO_CLASSE": "not_null",  # Não pode ser vazio (Substitui o dropna)
    "Data_Referencia": "not_null",    # Não pode ser vazio (Substitui o dropna)
    "Quantidade_Cotas_Emitidas": "decimal",         # Não pode conter letras
}

df_silver_fii_geral, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_silver_fii_geral,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_fii_geral_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_silver_fii_geral = df_silver_fii_geral.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_silver_fii_geral = df_silver_fii_geral.select(
    # 1. Identificação e Informações Básicas
    f.col('Tipo_Fundo_Classe').cast(t.StringType()).alias('tipo_fundo_classe'),
    f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()).alias('cnpj_fundo_classe'),
    f.col('Nome_Fundo_Classe').cast(t.StringType()).alias('nome_fundo_classe'),
    f.col('Data_Referencia').cast(t.DateType()).alias('data_referencia'),
    f.col('Versao').cast(t.IntegerType()).alias('versao'),
    f.col('Data_Entrega').cast(t.DateType()).alias('data_entrega'),
    f.col('Data_Funcionamento').cast(t.DateType()).alias('data_funcionamento'),
    f.col('Codigo_ISIN').cast(t.StringType()).alias('codigo_isin'),
    f.col('Quantidade_Cotas_Emitidas').cast(t.DecimalType(22, 2)).alias('quantidade_cotas_emitidas'),

    # 2. Características Estruturais do Fundo
    f.col('Publico_Alvo').cast(t.StringType()).alias('publico_alvo'),
    f.col('Fundo_Exclusivo').cast(t.StringType()).alias('fundo_exclusivo'),
    f.col('Cotistas_Vinculo_Familiar').cast(t.StringType()).alias('cotistas_vinculo_familiar'),
    f.col('Mandato').cast(t.StringType()).alias('mandato'),
    f.col('Segmento_Atuacao').cast(t.StringType()).alias('segmento_atuacao'),
    f.col('Tipo_Gestao').cast(t.StringType()).alias('tipo_gestao'),
    f.col('Prazo_Duracao').cast(t.StringType()).alias('prazo_duracao'),
    f.col('Data_Prazo_Duracao').cast(t.DateType()).alias('data_prazo_duracao'),
    f.col('Encerramento_Exercicio_Social').cast(t.StringType()).alias('encerramento_exercicio_social'),

    # 3. Negociação e Administração
    f.col('Mercado_Negociacao_Bolsa').cast(t.StringType()).alias('mercado_negociacao_bolsa'),
    f.col('Mercado_Negociacao_MBO').cast(t.StringType()).alias('mercado_negociacao_mbo'),
    f.col('Mercado_Negociacao_MB').cast(t.StringType()).alias('mercado_negociacao_mb'),
    f.col('Entidade_Administradora_BVMF').cast(t.StringType()).alias('entidade_administradora_bvmf'),
    f.col('Entidade_Administradora_CETIP').cast(t.StringType()).alias('entidade_administradora_cetip'),
    f.col('Nome_Administrador').cast(t.StringType()).alias('nome_administrador'),
    f.col('CNPJ_Administrador').cast(t.StringType()).alias('cnpj_administrador'),

    # 4. Endereço e Contatos do Administrador
    f.col('Logradouro').cast(t.StringType()).alias('logradouro'),
    f.col('Numero').cast(t.StringType()).alias('numero'),
    f.col('Complemento').cast(t.StringType()).alias('complemento'),
    f.col('Bairro').cast(t.StringType()).alias('bairro'),
    f.col('Cidade').cast(t.StringType()).alias('cidade'),
    f.col('Estado').cast(t.StringType()).alias('estado'),
    f.col('CEP').cast(t.StringType()).alias('cep'),
    f.col('Telefone1').cast(t.StringType()).alias('telefone1'),
    f.col('Telefone2').cast(t.StringType()).alias('telefone2'),
    f.col('Telefone3').cast(t.StringType()).alias('telefone3'),
    f.col('Site').cast(t.StringType()).alias('site'),
    f.col('Email').cast(t.StringType()).alias('email')
)

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_fundo_classe", "data_referencia"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_silver_fii_geral, 
    tabela_destino=SILVER_PATH, 
    chave_negocio=chave_negocio
    )